# ScortexIA/laurelia — TPU Training

Clona repo, entrena en TPU con flujo constante, sube a HF cada 10min con tag `gens0x`.

**Setup:**
1. Colab Secrets → `MY_KEY` = token HF (write)
2. Entorno → TPU v2-8

**Pipeline:**
1. Descarga primeros 60MB de Wikipedia → entrena BPE tokenizer → sube a HF si no existe
2. Entrena modelo en streaming desde `input.txt` (o MY_KEY si es URL)
3. Push cada 10 min a `ScortexIA/laurelia` @ `gens0x`

In [ ]:
# ── 1. Instalar ──
print('Installing...')
!pip install -q torch torchvision torch_xla[tpu] libtpu \
    safetensors tokenizers requests huggingface_hub \
    pyarrow datasets
print('Done.')

In [ ]:
# ── 2. Clonar/pull repo ──
import os
REPO = 'https://github.com/emanuelbertey/Evil-Inference-Code'
BRANCH = 'llamaburn'
DIR = '/content/Evil-Inference-Code'
if not os.path.exists(DIR):
    !git clone --branch {BRANCH} {REPO} {DIR}
else:
    !cd {DIR} && git pull origin {BRANCH}
%cd {DIR}

In [ ]:
# ── 3. MY_KEY ──
from google.colab import userdata
MY_KEY = userdata.get('mi_key') or ''
if MY_KEY:
    os.environ['MY_KEY'] = MY_KEY
    print('HF token loaded.')
else:
    print('WARNING: sin MY_KEY no se sube a HF.')

In [ ]:
# ── 4. Entrenar ──
import sys
sys.path.insert(0, 'rust')
from python_rust.train import train
train()

---
## Detalles

| Acción | Cada |
|---|---|
| Push a `ScortexIA/laurelia` @ `gens0x` | 10 minutos |
| Checkpoint local | 500 steps |
| Reanuda desde HF si existe | Al iniciar |

Tokenizer: BPE ByteLevel vocab 16000, entrenado en primeros 60MB de Wikipedia.
Modelo: 12.6M params, d_model=256, 6 layers, GQA (8/4), SwiGLU, RoPE.